# Data preprocessing
**Data source:** Manhattan Chocolate Society's [Flavors of Cacao](https://flavorsofcacao.com/chocolate_database.html) database.

In [ ]:
import pandas as pd

columns = [
    'REF', 'Company', 'Company Location', 'Review Date', 
    'Country of Bean Origin', 'Specific Bean Origin', 
    'Cocoa Percent', 'Ingredients', 'Most Memorable Characteristics', 'Rating'
]
df = pd.read_csv('data/data.txt', sep='\t', names=columns, skiprows=1)
df.head()


,REF,Company,Company Location,Review Date,Country of Bean Origin,Specific Bean Origin,Cocoa Percent,Ingredients,Most Memorable Characteristics,Rating
0,3016,Cacao Store,Japan,2026,Thailand,Chumphon,70%,"3- B,S,C","light apricot, pear, nutty, clean",3.25
1,3016,Cacao Store,Japan,2026,Taiwan,Fu-Wan,70%,"3- B,S,C","cocoa,roasty,pungent, chemical",2.75
2,3016,Cacao Store,Japan,2026,Madagascar,Ambanja,75%,"3- B,S,C","high acid, atypical honey",3.00
3,3016,URA,Peru,2026,Peru,"Piura, Chulucanas, Finca Tito Jimenez",70%,"3- B,S,C","grapes, roast (smoke like)",3.75
4,3020,URA,Peru,2026,Peru,"VRAEM 99, Pichari, Cusco, Finca Daria",70%,"3- B,S,C",plum,3.50


In [127]:
df = df[['Company Location', 'Review Date','Country of Bean Origin', 'Cocoa Percent',
       'Ingredients', 'Rating']]
df.head()
df.dropna()

,Company Location,Review Date,Country of Bean Origin,Cocoa Percent,Ingredients,Rating
0,Japan,2026,Thailand,70%,"3- B,S,C",3.25
1,Japan,2026,Taiwan,70%,"3- B,S,C",2.75
2,Japan,2026,Madagascar,75%,"3- B,S,C",3.00
3,Peru,2026,Peru,70%,"3- B,S,C",3.75
4,Peru,2026,Peru,70%,"3- B,S,C",3.50
...,...,...,...,...,...,...
1307,Puerto Rico,2016,U.S.A. (Puerto Rico),80%,"6-B,S,C,V,L,Sa",2.75
1308,Puerto Rico,2016,Dominican Republic,70%,"6-B,S,C,V,L,Sa",2.75
1309,Portugal,2016,Venezuela,76%,"2- B,S",2.50
1310,Portugal,2016,Jamaica,76%,"2- B,S",3.00


In [128]:

print('Company location:')
print(df['Company Location'].unique())

print('Bean origin:')
print(df['Country of Bean Origin'].unique())

Company location:
['Japan' 'Peru' 'Georgia' 'U.K.' 'Canada' 'Germany' 'U.S.A.' 'India'
 'France' 'Hungary' 'Spain' 'Costa Rica' 'Mexico' 'Belgium' 'Colombia'
 'China' 'Venezuela' 'El Salvador' 'Switzerland' 'Honduras' 'Thailand'
 'Jamaica' 'Italy' 'Benin' 'Lithuania' 'Denmark' 'Romania' 'Netherlands'
 'Malaysia' 'Estonia' 'Ireland' 'Vietnam' 'Sweden' 'Brazil' 'Nicaraqua'
 'Scotland' 'Singapore' 'Ecuador' 'Czech Republic' 'Sao Tome & Principe'
 'St. Lucia' 'Vanuatu' 'Taiwan' 'South Korea' 'Australia' 'Chile' 'U.A.E.'
 'Dominican Republic' 'New Zealand' 'Austria' 'Philippines' 'South Africa'
 'Iceland' 'Nicaragua' 'St.Vincent-Grenadines' 'Norway' 'Russia'
 'Suriname' 'Poland' 'Finland' 'Israel' 'Puerto Rico' 'Portugal']
Bean origin:
['Thailand' 'Taiwan' 'Madagascar' 'Peru' 'Philippines' 'India' 'U.S.A.'
 'Vanuatu' 'Tanzania' 'Indonesia' 'Venezuela' 'Bolivia' 'Guatemala'
 'Costa Rica' 'Belize' 'France (Martinique)' 'Vietnam' 'Mexico' 'Cuba'
 'U.S.A. (Puerto Rico)' 'Colombia' 'Uganda' 'Ecu

Some discrepancies, like spelling mistakes, in country naming have been found within the dataset. We also unified the countries naming by merging specific sub-regions into their respective countries where appropriate while treating distinct territories separately. We also removed Country of Bean Origin specified as 'Blend', since it is a mixture of cocoa from different countries.

In [129]:
company_fixes = {
    'Nicaraqua': 'Nicaragua',
    'St.Vincent-Grenadines': 'St. Vincent-Grenadines',
}

bean_fixes = {
    'Sao Tome': 'Sao Tome & Principe',
    'St.Vincent-Grenadines': 'St. Vincent-Grenadines',
    'Sulawesi': 'Indonesia',
    'Sumatra': 'Indonesia',
    'U.S.A. (Puerto Rico)': 'Puerto Rico',
    'France (Martinique)': 'Martinique',
    'France (Reunion)': 'Reunion'
}

df['Company Location'] = df['Company Location'].replace(company_fixes)
df['Country of Bean Origin'] = df['Country of Bean Origin'].replace(bean_fixes)
df = df[df['Country of Bean Origin'].str.lower() != 'blend'].copy()
df = df[df['Country of Bean Origin'].str.lower() != 'blend'].copy()


We removed entries for countries (both Cocoa Origin and Producer Company) that have less than 10 observations. The remaining country names were converted into integer indexes.

In [130]:
def print_smallest_entries(df, column_name, n=10):
    counts = df[column_name].value_counts()
    smallest = counts.nsmallest(n)
    
    print(f"Lowest number of entries in '{column_name}':")
    for item, count in smallest.items():
        print(f"{item}: {count} entries")

In [131]:
cond_company = df.groupby('Company Location')['Company Location'].transform('count') >= 10
cond_origin = df.groupby('Country of Bean Origin')['Country of Bean Origin'].transform('count') >= 10
df = df[cond_company & cond_origin].reset_index(drop=True)
excluded_locations = ['China', 'Taiwan']
excluded_origins = ['Thailand', 'Papua New Guinea']
df = df[~df['Company Location'].isin(excluded_locations)].copy()
df = df[~df['Country of Bean Origin'].isin(excluded_origins)].copy()

In [132]:
print_smallest_entries(df, 'Company Location', n=10)
print("\n")
print_smallest_entries(df, 'Country of Bean Origin', n=10)

Lowest number of entries in 'Company Location':
Switzerland: 11 entries
Brazil: 11 entries
Austria: 11 entries
Singapore: 13 entries
Peru: 14 entries
Australia: 14 entries
Venezuela: 14 entries
Colombia: 15 entries
New Zealand: 15 entries
Mexico: 17 entries


Lowest number of entries in 'Country of Bean Origin':
Costa Rica: 10 entries
Fiji: 10 entries
Solomon Islands: 10 entries
Indonesia: 12 entries
U.S.A.: 13 entries
Trinidad: 15 entries
Ghana: 16 entries
Uganda: 18 entries
Honduras: 20 entries
Philippines: 22 entries


In [133]:
countries  = sorted(df['Country of Bean Origin'].unique())
locations  = sorted(df['Company Location'].unique())
country_idx  = {c: i+1 for i, c in enumerate(countries)}
location_idx = {l: i+1 for i, l in enumerate(locations)}
df['Bean Origin ID']  = df['Country of Bean Origin'].map(country_idx)
df['Company Location ID'] = df['Company Location'].map(location_idx)

Number of ingridients has been derived from Ingredients string column. The cocoa percent value has been converted into float and normalized by centering it around the mean value. We also intruduced binary 'Has vanilla' factor, derived from the Ingredients column. The missing values have been filled with mean value.

In [134]:
empty_indices = df[df['Ingredients'].astype(str).str.strip() == ''].index
df.loc[empty_indices, 'Ingredients'] = '3- B'
df['Ingredients'] = df['Ingredients'].replace(r'^\s*$', pd.NA, regex=True)

df['Number of ingredients'] = df['Ingredients'].astype(str).str.extract(r"(\d+)-").astype(float)

df['Number of ingredients norm'] = df['Number of ingredients'].fillna(df['Number of ingredients'].median())
df['Number of ingredients norm'] = (df['Number of ingredients'] - df['Number of ingredients'].mean()) / df['Number of ingredients'].std()

df['Has vanilla'] = df['Ingredients'].astype(str).str.contains("V", na=False).astype(int)
df["Year Norm"] = (df["Review Date"] - df["Review Date"].mean()) / df["Review Date"].std()

In [135]:
df['Cocoa Percent norm'] = df['Cocoa Percent'].astype(str).str.replace("%", "").astype(float)
cocoa_mean = df['Cocoa Percent norm'].mean()
cocoa_std = df['Cocoa Percent norm'].std()
df['Cocoa Percent norm'] = (df['Cocoa Percent norm'] - cocoa_mean)/cocoa_std

In [136]:
df.drop('Ingredients', axis='columns', inplace=True)

In [137]:
df.head()

,Company Location,Review Date,Country of Bean Origin,Cocoa Percent,Rating,Bean Origin ID,Company Location ID,Number of ingredients,Number of ingredients norm,Has vanilla,Year Norm,Cocoa Percent norm
1,Japan,2026,Madagascar,75%,3.00,15,11,3.0,0.345818,0,2.447873,0.973354
2,Peru,2026,Peru,70%,3.75,18,15,3.0,0.345818,0,2.447873,-0.412919
3,Peru,2026,Peru,70%,3.50,18,15,3.0,0.345818,0,2.447873,-0.412919
4,U.K.,2026,Philippines,70%,3.50,19,19,3.0,0.345818,0,2.447873,-0.412919
5,Canada,2026,Peru,80%,3.25,18,5,3.0,0.345818,0,2.447873,2.359627


In [138]:
df.describe()

,Review Date,Rating,Bean Origin ID,Company Location ID,Number of ingredients,Number of ingredients norm,Has vanilla,Year Norm,Cocoa Percent norm
count,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000,1.076000e+03,1076.000000,1.076000e+03,1.076000e+03
mean,2019.517658,3.246980,13.544610,15.455390,2.768587,7.924268e-17,0.027881,-1.341843e-14,-2.707458e-16
std,2.648152,0.369698,7.577498,6.266252,0.669174,1.000000e+00,0.164709,1.000000e+00,1.000000e+00
min,2016.000000,2.250000,1.000000,1.000000,2.000000,-1.148561e+00,0.000000,-1.328344e+00,-3.185465e+00
25%,2017.000000,3.000000,6.000000,10.000000,2.000000,-1.148561e+00,0.000000,-9.507225e-01,-4.129187e-01
50%,2019.000000,3.250000,15.000000,20.000000,3.000000,3.458184e-01,0.000000,-1.954789e-01,-4.129187e-01
75%,2021.000000,3.500000,18.000000,20.000000,3.000000,3.458184e-01,0.000000,5.597646e-01,4.188451e-01
max,2026.000000,4.000000,26.000000,21.000000,6.000000,4.828958e+00,1.000000,2.447873e+00,5.132174e+00


In [139]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1076 entries, 1 to 1103
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Company Location            1076 non-null   object 
 1   Review Date                 1076 non-null   int64  
 2   Country of Bean Origin      1076 non-null   object 
 3   Cocoa Percent               1076 non-null   object 
 4   Rating                      1076 non-null   float64
 5   Bean Origin ID              1076 non-null   int64  
 6   Company Location ID         1076 non-null   int64  
 7   Number of ingredients       1076 non-null   float64
 8   Number of ingredients norm  1076 non-null   float64
 9   Has vanilla                 1076 non-null   int64  
 10  Year Norm                   1076 non-null   float64
 11  Cocoa Percent norm          1076 non-null   float64
dtypes: float64(5), int64(4), object(3)
memory usage: 141.6+ KB


In [ ]:
df.to_csv('data/chocolate_data_preprocessed.csv', index=False)